In [1]:
# NetCDF Dimension Renamer - Jupyter Notebook
# This notebook renames dimensions in a netCDF file by creating a new file
# Tested with GLODAOv2_2016b data files for temperature and salinity as per REF
# Version 1.1 PSmith 25-12-17

# Cell 1: Import libraries
import netCDF4 as nc
from pathlib import Path
import shutil

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ============================================================
# Cell 2: Configure your file and settings
# ============================================================

# Define your input file path
input_file = "/inputs/NOAA-NCEI/GLODAP-2-2016b/GLODAPv2.2016b.salinity.nc"  # <-- CHANGE THIS to your input file path

# Define your output file path (will be created)
# output_file = "GLODAPv2.2016c.temperature.nc"  # <-- CHANGE THIS to your output file path
output_file = "GLODAPv2.2016c.salinity.nc"

# Define the dimension you want to rename
old_dimension_name = "depth_surface"  # <-- CHANGE THIS to the dimension to rename
new_dimension_name = "depth"  # <-- CHANGE THIS to the new dimension name

print(f"Configuration set:")
print(f"  Input file: {input_file}")
print(f"  Output file: {output_file}")
print(f"  Rename dimension: '{old_dimension_name}' → '{new_dimension_name}'")

Configuration set:
  Input file: ../inputs/NOAA-NCEI/GLODAP-2-2016b/GLODAPv2.2016b.salinity.nc
  Output file: GLODAPv2.2016c.salinity.nc
  Rename dimension: 'depth_surface' → 'depth'


In [3]:
# ============================================================
# Cell 3: Check if files exist and validate
# ============================================================

if not Path(input_file).exists():
    print(f"✗ Error: Input file not found: {input_file}")
    print("Please update the input_file in Cell 2")
else:
    print(f"✓ Input file found: {input_file}")

if Path(output_file).exists():
    print(f"⚠ Warning: Output file already exists: {output_file}")
    print("It will be overwritten if you proceed!")
else:
    print(f"✓ Output file will be created: {output_file}")

✓ Input file found: /obs4MIPs-cmor-tables/inputs/NOAA-NCEI/GLODAP-2-2016b/GLODAPv2.2016b.salinity.nc
✓ Output file will be created: GLODAPv2.2016c.salinity.nc


In [4]:
# ============================================================
# Cell 4: View current dataset structure
# ============================================================

try:
    with nc.Dataset(input_file, 'r') as ds:
        print("\n=== Current Dataset Structure ===\n")
        
        print("Dimensions:")
        for dim_name, dim in ds.dimensions.items():
            size = len(dim) if not dim.isunlimited() else "UNLIMITED"
            marker = " <-- TARGET" if dim_name == old_dimension_name else ""
            print(f"  {dim_name}: {size}{marker}")
        
        print("\nVariables (dimensions):")
        for var_name, var in ds.variables.items():
            dims = var.dimensions
            marker = " <-- WILL BE AFFECTED" if old_dimension_name in dims else ""
            print(f"  {var.dtype} {var_name}{dims}{marker}")
        
        print(f"\nGlobal Attributes: {len(ds.ncattrs())}")
        
        # Check if the dimension exists
        if old_dimension_name not in ds.dimensions:
            print(f"\n✗ Error: Dimension '{old_dimension_name}' not found in file!")
        else:
            print(f"\n✓ Dimension '{old_dimension_name}' found and ready to rename")
            
except FileNotFoundError:
    print(f"Error: Cannot open file '{input_file}'")
except Exception as e:
    print(f"Error reading file: {e}")



=== Current Dataset Structure ===

Dimensions:
  lon: 360
  lat: 180
  depth_surface: 33 <-- TARGET
  time: 1
  snr: 1

Variables (dimensions):
  float64 lon('lon',)
  float64 lat('lat',)
  float64 salinity('depth_surface', 'lat', 'lon') <-- WILL BE AFFECTED
  float64 salinity_error('depth_surface', 'lat', 'lon') <-- WILL BE AFFECTED
  float64 Input_mean('depth_surface', 'lat', 'lon') <-- WILL BE AFFECTED
  float64 Input_std('depth_surface', 'lat', 'lon') <-- WILL BE AFFECTED
  float64 Input_N('depth_surface', 'lat', 'lon') <-- WILL BE AFFECTED
  float64 salinity_relerr('depth_surface', 'lat', 'lon') <-- WILL BE AFFECTED
  float64 SnR('snr',)
  float64 CL('snr',)
  float64 Depth('depth_surface',) <-- WILL BE AFFECTED

Global Attributes: 6

✓ Dimension 'depth_surface' found and ready to rename


In [5]:
# ============================================================
# Cell 5: Perform the dimension rename (CREATE NEW FILE)
# ============================================================

try:
    print(f"Renaming dimension '{old_dimension_name}' to '{new_dimension_name}'...\n")
    
    # Open input file for reading
    with nc.Dataset(input_file, 'r') as src:
        
        # Check if old dimension exists
        if old_dimension_name not in src.dimensions:
            raise ValueError(f"Dimension '{old_dimension_name}' not found in input file")
        
        # Create output file
        with nc.Dataset(output_file, 'w', format=src.data_model) as dst:
            
            # Copy global attributes
            print("Copying global attributes...")
            dst.setncatts({attr: src.getncattr(attr) for attr in src.ncattrs()})
            
            # Copy dimensions (with rename)
            print("Copying dimensions...")
            for name, dimension in src.dimensions.items():
                new_name = new_dimension_name if name == old_dimension_name else name
                dst.createDimension(
                    new_name,
                    len(dimension) if not dimension.isunlimited() else None
                )
                print(f"  {name} → {new_name} (size: {len(dimension)})")
            
            # Copy variables (with updated dimension names)
            print("\nCopying variables...")
            for name, variable in src.variables.items():
                # Update dimension names in variable
                new_dims = tuple(
                    new_dimension_name if dim == old_dimension_name else dim
                    for dim in variable.dimensions
                )
                
                # Create variable in destination
                dst_var = dst.createVariable(
                    name,
                    variable.datatype,
                    new_dims,
                    fill_value=variable._FillValue if hasattr(variable, '_FillValue') else None
                )
                
                # Copy variable attributes
                dst_var.setncatts({
                    attr: variable.getncattr(attr) 
                    for attr in variable.ncattrs() 
                    if attr != '_FillValue'
                })
                
                # Copy variable data
                dst_var[:] = variable[:]
                
                print(f"  {name}{variable.dimensions} → {name}{new_dims}")
    
    print(f"\n✓ Successfully created: {output_file}")
    print(f"✓ Dimension '{old_dimension_name}' renamed to '{new_dimension_name}'")
    
except PermissionError:
    print(f"Error: Permission denied. Files may be open in another program.")
except Exception as e:
    print(f"Error during rename: {e}")
    import traceback
    traceback.print_exc()

Renaming dimension 'depth_surface' to 'depth'...

Copying global attributes...
Copying dimensions...
  lon → lon (size: 360)
  lat → lat (size: 180)
  depth_surface → depth (size: 33)
  time → time (size: 1)
  snr → snr (size: 1)

Copying variables...
  lon('lon',) → lon('lon',)
  lat('lat',) → lat('lat',)
  salinity('depth_surface', 'lat', 'lon') → salinity('depth', 'lat', 'lon')
  salinity_error('depth_surface', 'lat', 'lon') → salinity_error('depth', 'lat', 'lon')
  Input_mean('depth_surface', 'lat', 'lon') → Input_mean('depth', 'lat', 'lon')
  Input_std('depth_surface', 'lat', 'lon') → Input_std('depth', 'lat', 'lon')
  Input_N('depth_surface', 'lat', 'lon') → Input_N('depth', 'lat', 'lon')
  salinity_relerr('depth_surface', 'lat', 'lon') → salinity_relerr('depth', 'lat', 'lon')
  SnR('snr',) → SnR('snr',)
  CL('snr',) → CL('snr',)
  Depth('depth_surface',) → Depth('depth',)

✓ Successfully created: GLODAPv2.2016c.salinity.nc
✓ Dimension 'depth_surface' renamed to 'depth'


In [6]:
# ============================================================
# Cell 6: Verify the output file
# ============================================================

try:
    with nc.Dataset(output_file, 'r') as ds:
        print("\n=== Output File Structure ===\n")
        
        print("Dimensions:")
        for dim_name, dim in ds.dimensions.items():
            size = len(dim) if not dim.isunlimited() else "UNLIMITED"
            marker = " <-- RENAMED!" if dim_name == new_dimension_name else ""
            print(f"  {dim_name}: {size}{marker}")
        
        print("\nVariables (dimensions):")
        for var_name, var in ds.variables.items():
            dims = var.dimensions
            marker = " <-- UPDATED" if new_dimension_name in dims else ""
            print(f"  {var.dtype} {var_name}{dims}{marker}")
        
        print(f"\nGlobal Attributes: {len(ds.ncattrs())}")
        
        # Verify the change
        if new_dimension_name in ds.dimensions and old_dimension_name not in ds.dimensions:
            print(f"\n✓ Success! Dimension successfully renamed")
        else:
            print(f"\n⚠ Warning: Something unexpected happened")
            
except FileNotFoundError:
    print(f"Error: Output file not found: {output_file}")
except Exception as e:
    print(f"Error reading output file: {e}")



=== Output File Structure ===

Dimensions:
  lon: 360
  lat: 180
  depth: 33 <-- RENAMED!
  time: 1
  snr: 1

Variables (dimensions):
  float64 lon('lon',)
  float64 lat('lat',)
  float64 salinity('depth', 'lat', 'lon') <-- UPDATED
  float64 salinity_error('depth', 'lat', 'lon') <-- UPDATED
  float64 Input_mean('depth', 'lat', 'lon') <-- UPDATED
  float64 Input_std('depth', 'lat', 'lon') <-- UPDATED
  float64 Input_N('depth', 'lat', 'lon') <-- UPDATED
  float64 salinity_relerr('depth', 'lat', 'lon') <-- UPDATED
  float64 SnR('snr',)
  float64 CL('snr',)
  float64 Depth('depth',) <-- UPDATED

Global Attributes: 6

✓ Success! Dimension successfully renamed


In [7]:
# ============================================================
# Cell 7 (Optional): Compare input and output files
# ============================================================

try:
    print("=== File Comparison ===\n")
    
    with nc.Dataset(input_file, 'r') as src, nc.Dataset(output_file, 'r') as dst:
        print(f"Input dimensions:  {list(src.dimensions.keys())}")
        print(f"Output dimensions: {list(dst.dimensions.keys())}")
        
        print(f"\nInput variables:  {list(src.variables.keys())}")
        print(f"Output variables: {list(dst.variables.keys())}")
        
        # Check file sizes
        input_size = Path(input_file).stat().st_size
        output_size = Path(output_file).stat().st_size
        print(f"\nInput file size:  {input_size:,} bytes")
        print(f"Output file size: {output_size:,} bytes")
        
except Exception as e:
    print(f"Error comparing files: {e}")

=== File Comparison ===

Input dimensions:  ['lon', 'lat', 'depth_surface', 'time', 'snr']
Output dimensions: ['lon', 'lat', 'depth', 'time', 'snr']

Input variables:  ['lon', 'lat', 'salinity', 'salinity_error', 'Input_mean', 'Input_std', 'Input_N', 'salinity_relerr', 'SnR', 'CL', 'Depth']
Output variables: ['lon', 'lat', 'salinity', 'salinity_error', 'Input_mean', 'Input_std', 'Input_N', 'salinity_relerr', 'SnR', 'CL', 'Depth']

Input file size:  102,664,378 bytes
Output file size: 102,666,553 bytes
